# Week 39, Day 3: Smell: random expansion and sparse codes

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w039/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Concentration invariance by normalisation

Write normalise(v): divide the vector by its mean (divisive normalisation), then subtract the mean of the result (centring), returning a vector whose pattern, not its overall level, carries the odour's identity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2017)
d = 50
odour = r.exponential(1, d)                              # affinities of the 50 receptor types for one odour

def orn(affinity, conc, K=200.0):
    x = conc * affinity
    return 100 * x / (x + K)                             # saturating receptor responses

def normalise(v):
    ...

for conc in (1, 10, 100):
    raw = orn(odour, conc); nv = normalise(raw)
    print(f"concentration {conc:3d}: raw length {np.linalg.norm(raw):7.1f}; normalised, first five channels {np.round(nv[:5], 2)}")

In [ ]:
import numpy as np
def test_raw():
    "the raw ORN vector's length changes more than tenfold over a 100-fold concentration range"
    assert np.linalg.norm(orn(odour, 100)) / np.linalg.norm(orn(odour, 1)) > 10

def test_normalised():
    "after normalisation, the patterns at 1 and 100 units of concentration have cosine similarity above 0.85"
    a, b = normalise(orn(odour, 1)), normalise(orn(odour, 100))
    assert np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)) > 0.85

test_raw()
test_normalised()
print("All checks passed")

### Exercise: The fly hash against LSH

Write fly_hash(X, m=2000, k=6, top=20): a binary m by d connection matrix in which each Kenyon cell samples k random inputs, the projection X @ M, and a winner-take-all that sets the top entries of each row to 1 and the rest to 0. The neighbour-precision measure and sign LSH are given.

In [ ]:
centres = r.exponential(1, (40, d))
X = np.repeat(centres, 25, 0) * r.lognormal(0, 0.4, (1000, d))       # 40 families of 25 related odours
X = np.array([normalise(x) for x in X])

def fly_hash(X, m=2000, k=6, top=20, seed=0):
    g = np.random.default_rng(seed)
    M = np.zeros((X.shape[1], m))
    for j in range(m):
        M[g.choice(X.shape[1], k, replace=False), j] = 1
    Y = X @ M
    H = ...
    return H

def sign_lsh(X, bits=20, seed=0):
    g = np.random.default_rng(seed)
    return np.sign(X @ g.normal(size=(X.shape[1], bits)))

def neighbour_precision(X, H, n_query=200, k=25):
    sq = lambda A: (A ** 2).sum(1)[:, None] + (A ** 2).sum(1)[None] - 2 * A @ A.T
    dx, dh = sq(X), sq(H)
    hits = [len(set(np.argsort(dx[i])[1:k + 1]) & set(np.argsort(dh[i], kind="stable")[1:k + 1])) / k for i in range(n_query)]
    return np.mean(hits)

prec = {}
for tags in (20, 50):
    prec["fly", tags] = neighbour_precision(X, fly_hash(X, top=tags))
    prec["LSH", tags] = neighbour_precision(X, sign_lsh(X, bits=tags))
    print(f"{tags} tags: fly hash {prec['fly', tags]:.2f}, sign LSH {prec['LSH', tags]:.2f}")

In [ ]:
import numpy as np
def test_fly_wins_small():
    "with 20 tags, the fly hash finds at least 15 percentage points more true neighbours than sign LSH"
    assert prec["fly", 20] > prec["LSH", 20] + 0.15

def test_both_good():
    "with 50 tags, both find more than 70% of the true neighbours"
    assert prec["fly", 50] > 0.7 and prec["LSH", 50] > 0.7

test_fly_wins_small()
test_both_good()
print("All checks passed")

### Exercise: Sweeping the number of inputs

For k = 1, 3, 6, 12, 25 and 50 inputs per Kenyon cell (with 20 tags), compute the neighbour precision of the fly hash and store it in deg[k].

In [ ]:
deg = {}
for k in (1, 3, 6, 12, 25, 50):
    deg[k] = ...
    print(f"{k:2d} inputs per cell: precision {deg[k]:.2f}")
plt.plot(list(deg), list(deg.values()), "o-"); plt.xscale("log"); plt.xlabel("inputs per Kenyon cell"); plt.ylabel("neighbour precision"); plt.show()

In [ ]:
import numpy as np
def test_one():
    "one input per cell performs far worse than six"
    assert deg[1] < deg[6] - 0.3

def test_all():
    "sampling all 50 inputs collapses the code"
    assert deg[50] < 0.3

test_one()
test_all()
print("All checks passed")